In [1]:
import folium
import pandas as pd
from folium.plugins import MarkerCluster
from folium.features import DivIcon
import math

# 1. Đọc dữ liệu đã xử lý từ Bước 3
df = pd.read_csv("dataset_part_2.csv")

# Chỉ lấy các cột cần thiết cho bản đồ
df_map = df[['LaunchSite', 'Latitude', 'Longitude', 'Class']]
launch_sites_df = df_map.groupby(['LaunchSite'], as_index=False).first()

# Khởi tạo bản đồ Folium với tâm ở trung tâm nước Mỹ
nasa_coordinate = [29.5528040171, -95.0979848525]
site_map = folium.Map(location=nasa_coordinate, zoom_start=4)

# 2. ĐÁNH DẤU CÁC BỆ PHÓNG (LAUNCH SITES)
# Duyệt qua từng bệ phóng để thêm Marker vòng tròn và nhãn tên
for index, row in launch_sites_df.iterrows():
    coordinate = [row['Latitude'], row['Longitude']]
    
    # Vẽ vòng tròn bao quanh bệ phóng
    circle = folium.Circle(coordinate, radius=1000, color='#d35400', fill=True).add_child(folium.Popup(row['LaunchSite']))
    site_map.add_child(circle)
    
    # Tạo nhãn tên bệ phóng hiển thị trực tiếp trên bản đồ
    marker = folium.map.Marker(
        coordinate,
        icon=DivIcon(
            icon_size=(20,20),
            icon_anchor=(0,0),
            html=f'<div style="font-size: 12pt; color:#d35400;"><b>{row["LaunchSite"]}</b></div>',
        )
    )
    site_map.add_child(marker)

# 3. ĐÁNH DẤU CÁC CHUYẾN BAY THÀNH CÔNG/THẤT BẠI (MARKER CLUSTER)
marker_cluster = MarkerCluster()
site_map.add_child(marker_cluster)

# Thêm từng lần phóng vào cluster
for index, row in df_map.iterrows():
    coordinate = [row['Latitude'], row['Longitude']]
    
    # Đặt màu sắc: Xanh lá nếu hạ cánh thành công (Class=1), Đỏ nếu thất bại (Class=0)
    marker_color = 'green' if row['Class'] == 1 else 'red'
    popup_text = f"Launch Site: {row['LaunchSite']}\nStatus: {'Success' if row['Class'] == 1 else 'Fail'}"
    
    marker = folium.Marker(
        coordinate,
        icon=folium.Icon(color='white', icon_color=marker_color),
        popup=popup_text
    )
    marker_cluster.add_child(marker)

# 4. PHÂN TÍCH KHOẢNG CÁCH (PROXIMITY ANALYSIS)
# Hàm tính khoảng cách giữa 2 điểm tọa độ (Sử dụng công thức Haversine)
def calculate_distance(lat1, lon1, lat2, lon2):
    R = 6373.0  # Bán kính Trái Đất (km)
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = math.sin(dlat / 2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    return R * c

# Thí nghiệm đo khoảng cách từ bệ phóng KSC LC-39A (28.573255, -80.646895) tới các điểm lân cận:
launch_site_lat = 28.573255
launch_site_lon = -80.646895

# Tọa độ các điểm cần đo:
points = {
    "Coastline (Bờ biển)": [28.573255, -80.6068],
    "Highway (Đường cao tốc)": [28.573255, -80.6554],
    "Railway (Đường sắt)": [28.573255, -80.6542],
    "City (Thành phố Titusville)": [28.6115, -80.8077]
}

# Đo và vẽ các đường kết nối lên bản đồ
for key, point_coord in points.items():
    distance = calculate_distance(launch_site_lat, launch_site_lon, point_coord[0], point_coord[1])
    
    # Tạo nhãn hiển thị khoảng cách (km) tại điểm đích
    distance_marker = folium.Marker(
        point_coord,
        icon=DivIcon(
            icon_size=(20,20),
            icon_anchor=(0,0),
            html=f'<div style="font-size: 10pt; color:#2c3e50;"><b>{distance:.2f} KM</b></div>',
        )
    )
    site_map.add_child(distance_marker)
    
    # Vẽ đường thẳng đứt quãng (polyline) nối từ bệ phóng tới điểm đích
    lines = folium.PolyLine(locations=[[launch_site_lat, launch_site_lon], point_coord], weight=2, color='blue', dash_array='5, 5')
    site_map.add_child(lines)

# 5. Lưu bản đồ thành file HTML
site_map.save("spacex_launches_map.html")
print("Tạo bản đồ thành công! Bản đồ đã được lưu tại file 'spacex_launches_map.html'")

Tạo bản đồ thành công! Bản đồ đã được lưu tại file 'spacex_launches_map.html'
